# Precursor frequency for an epitope

How much of a repertoire can see a given pMHC? The estimand is

$$F(e) = \sum_{\tau \in C_e} \pi(\tau)$$

— the probability that a random naive-repertoire junction recognises epitope $e$. This is the
continuous quantity behind the word "immunogenic".

This notebook walks the three things that make it harder than summing `Pgen` over the TCRs a
database happens to hold:

1. **The values span orders of magnitude.** A mean is meaningless; the sum is set by a handful of
   public clonotypes.
2. **The neighbourhoods overlap.** Cognate junctions are near-duplicates by construction, so adding
   their per-sequence ball masses double-counts the shared region. `union_mass` is exact and does
   not enumerate.
3. **Most of the cognate set was never catalogued.** Two answers, because they are different
   questions: a finite census of the neighbourhood, and a Horvitz–Thompson extrapolation whose
   *mass* converges but whose *count* does not.

Needs the optional extra:

```
pip install 'vdjmatch[precursor]'
```

In [1]:
# Environment: record the versions every number below depends on.
from importlib.metadata import version

import polars as pl

import vdjmatch
from vdjmatch import db, precursor as P

SEED = 0  # nothing here samples, but the VDJdb release pin is the reproducibility knob
VDJDB_TAG = "2026-06-11-ZENODO"
for pkg in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(f"{pkg:10s} {version(pkg)}")

vdjmatch   0.1.2
vdjtools   3.9.3
seqtree    0.7.0
polars     1.41.2


## 1. Load a cognate set

**CDR3 is not junction.** VDJdb's column is *named* `cdr3` but holds junctions — Cys104 and
Phe118/Trp included — which is what the recombination model wants. An anchor-stripped IMGT CDR3
scores exactly `0.0` with no error, so `check_junctions` exists to make that failure loud.

In [2]:
# One well-sampled HLA-A*02:01 epitope: influenza A M1 58-66.
EPITOPE = "GILGFVFTL"

vdj = db.load(asset="slim", species="HomoSapiens", pin=VDJDB_TAG)
beta = vdj.filter((pl.col("gene") == "TRB") & (pl.col("epitope") == EPITOPE))

junctions, suspect = P.check_junctions(beta["cdr3"].unique().to_list())
junctions = list(dict.fromkeys(junctions))
print(f"{EPITOPE}: {beta.height:,} records, {len(junctions):,} distinct junctions, "
      f"{len(suspect)} failed the anchor check")

GILGFVFTL: 6,629 records, 6,627 distinct junctions, 2 failed the anchor check


## 2. The spread is the point

`observed_mass` is the sum of `Pgen` over the recorded junctions — a **strict lower bound** on
$F(e)$, and a biased one: a TCR enters a specificity database roughly in proportion to its
repertoire frequency, so the recorded members are systematically the high-`Pgen` ones.

In [3]:
# Per-junction Pgen, then how far apart the values are and how concentrated their sum is.
import numpy as np

model = P.load_model("TRB")
pg = np.array(P.pgen(model, junctions))
nz = np.sort(pg[pg > 0])[::-1]

print(f"observed_mass          {P.observed_mass(model, junctions):.3e}")
print(f"median Pgen            {np.median(nz):.3e}")
print(f"spread                 {np.log10(nz[0] / nz[-1]):.1f} orders of magnitude")
print(f"top-1  share of sum    {nz[0] / nz.sum():.3f}")
print(f"top-10 share of sum    {nz[:10].sum() / nz.sum():.3f}")
print(f"Pgen == 0 under model  {(pg == 0).sum()}")

observed_mass          3.305e-04
median Pgen            1.724e-09
spread                 30.0 orders of magnitude
top-1  share of sum    0.010
top-10 share of sum    0.080
Pgen == 0 under model  3


## 3. The union, not the sum

One substitution is part of the estimator, not a tuned parameter: at radius 0 a repertoire is too
sparse for either route to be estimable. But two cognate junctions one substitution apart have balls
sharing 20 sequences, and adding their masses counts that region twice.

`union_mass` is exact without enumerating the union. Every member is counted $\mathrm{cov}(x)$ times
by the naive sum, so

$$m\Big(\bigcup_a B_r(a)\Big) = \sum_a m\big(B_r(a)\big) - \sum_{x\,:\,\mathrm{cov}(x)\ge 2} \big(\mathrm{cov}(x)-1\big)\,P_{\mathrm{gen}}(x)$$

with no inclusion–exclusion and hence no truncation error. Only centres inside one connected
component of the $2r$ graph can contribute, so singleton components cost nothing.

In [4]:
# The exact union, and the double-counting a naive sum would have invented.
import time

t0 = time.perf_counter()
u = P.union_mass(model, junctions, r=1)
dt = time.perf_counter() - t0

print(f"naive per-sequence sum   {u['naive_sum']:.3e}")
print(f"exact union              {u['union']:.3e}")
print(f"double-counting avoided  {u['overlap']:.1%}")
print(f"components               {u['n_components']:,} "
      f"({u['n_clustered']:,} of {u['n_seqs']:,} junctions have a close neighbour)")
print(f"Pgen calls for the correction: {u['n_multiply_covered']:,} "
      f"(the union itself holds {u['n_union']:,})   [{dt:.1f}s]")

naive per-sequence sum   1.257e-02
exact union              7.432e-03
double-counting avoided  40.9%
components               2,968 (3,897 of 6,627 junctions have a close neighbour)
Pgen calls for the correction: 42,928 (the union itself holds 1,709,201)   [39.5s]


In [5]:
# Regression against the oracle: enumerate the union and score every member. Slow, and exact.
small = junctions[:150]
fast, oracle = P.union_mass(model, small, r=1), P.ball_mass(model, small, r=1)
rel = abs(fast["union"] - oracle["union"]) / oracle["union"]
print(f"union_mass {fast['union']:.9e}\nball_mass  {oracle['union']:.9e}\nrelative   {rel:.2e}")

union_mass 2.278723889e-04
ball_mass  2.278723889e-04
relative   1.19e-16


## 4. The ball is a smoother, not a coverage correction

It is tempting to read the neighbourhood mass as "the observed mass plus what sampling missed". It
is not. The ball is a statement about **cognacy**: a junction one substitution from a cognate TCR is
itself cognate with some probability, measured by Mayer & Callan (*PNAS* 2023;120:e2213264120) to
fall about ten-fold per unit of edit distance. `shell_profile` applies that per shell.

Shells are obtained by *differencing unions*, which is exact because min-distance shells partition
the ball — so the shell masses inherit `union_mass`'s exactness and nothing is enumerated to get
them. What still has to be enumerated is the multiply-covered set, and only within a connected
component. **That is where the real ceiling is**, and on a well-sampled epitope at radius 2 it is
reached: this cognate set has one component of a couple of thousand junctions whose radius-2 union
runs to tens of millions of sequences. The estimator refuses loudly rather than thrashing, and says
what to do about it.

In [6]:
# Shell-resolved mass with cognacy retention alpha = 0.1 per edit, at the recommended radius.
prof = P.shell_profile(model, junctions, r=1, alpha=0.1)
for s in prof["shells"]:
    print(f"  shell r={s['r']}  weight {s['alpha']:.3f}  n {s['n']:>9,}  mass {s['mass']:.3e}")
print(f"\nraw union (alpha=1)   {prof['union']:.3e}")
print(f"retained (alpha=0.1)  {prof['retained']:.3e}")
print(f"observed bound        {P.observed_mass(model, junctions):.3e}")

# Radius 2 on the whole set hits the component ceiling. That is the documented behaviour, not a
# crash: components are independent, so their union masses add exactly and splitting loses nothing.
try:
    P.shell_profile(model, junctions, r=2, alpha=0.1)
except MemoryError as e:
    print(f"\nr=2 on all {len(junctions):,} junctions ->\n  {e}")

# On a subset small enough to fit, the radius-2 profile runs and shell 2 is the dominant volume.
prof2 = P.shell_profile(model, junctions[:200], r=2, alpha=0.1)
print("\nradius-2 profile on 200 of them:")
for s in prof2["shells"]:
    print(f"  shell r={s['r']}  weight {s['alpha']:.3f}  n {s['n']:>9,}  mass {s['mass']:.3e}")
print(f"  retained {prof2['retained']:.3e} against a raw union of {prof2['union']:.3e}")

  shell r=0  weight 1.000  n     6,627  mass 3.305e-04
  shell r=1  weight 0.100  n 1,702,574  mass 7.101e-03

raw union (alpha=1)   7.432e-03
retained (alpha=0.1)  1.041e-03


observed bound        3.305e-04



r=2 on all 6,627 junctions ->
  one connected component of 1199 junctions has a radius-2 union of 43,726,919 sequences (~8.3 GB as Python strings), above max_members=20,000,000. Raise max_members if you have the memory, split this group, or lower r; components are independent, so their union masses add exactly and splitting by component loses nothing.



radius-2 profile on 200 of them:
  shell r=0  weight 1.000  n       200  mass 8.251e-06
  shell r=1  weight 0.100  n    53,380  mass 3.327e-04
  shell r=2  weight 0.010  n 6,707,831  mass 4.134e-03
  retained 8.287e-05 against a raw union of 4.475e-03


## 5. What was never catalogued

Two answers, and they are not interchangeable.

**The ball census** is finite and exact given the ball: `n_union - n_observed` sequences sit in the
neighbourhood of a known cognate TCR and appear in no database.

**Horvitz–Thompson** extrapolates instead, using the fact that `Pgen` *is* the sampling probability,
so the inclusion probability $\pi(j) = 1 - e^{-N p_j}$ is known rather than fitted. Good–Turing is
the wrong tool here — Laydon et al. (*PLoS Comput Biol* 2014;10:e1003646) measure 61.7% median
error for it on real TCR abundance data, because the capture-probability distribution is far too
heterogeneous for the uniform-multinomial assumption behind it.

The **mass** converges, because the weight $p/\pi \to 1/N$ as $p \to 0$. The **count** does not,
because $1/\pi$ diverges — so `richness_reliable` says when the extrapolated count has stopped
meaning anything.

In [7]:
# Capture units: how many distinct studies re-reported each junction for this epitope.
mult = (beta.group_by("cdr3").agg(pl.col("reference_id").n_unique().alias("m"))
            .filter(pl.col("cdr3").is_in(junctions)))

un = P.unseen_junctions(model, mult["cdr3"].to_list(), mult["m"].to_list(),
                        n_units=beta["reference_id"].n_unique())
print(f"ball census:  {u['n_union'] - u['n_seqs']:,} uncatalogued candidates "
      f"carrying mass {prof['union'] - P.observed_mass(model, junctions):.3e}")
if un["degenerate"]:
    print(f"Horvitz-Thompson: declined -- {un['reason']}")
else:
    print(f"Horvitz-Thompson: {un['n_unseen']:,.0f} unseen carrying {un['unseen_mass']:.3e}; "
          f"an average unseen junction is {un['rarity_ratio']:,.0f}x rarer than an observed one")
    print(f"                  richness_reliable = {un['richness_reliable']} "
          f"(min inclusion {un['min_inclusion']:.3g})")

ball census:  1,702,574 uncatalogued candidates carrying mass 7.101e-03
Horvitz-Thompson: declined -- every junction is a singleton: the capture curve is unidentified (theta -> 0, the Horvitz-Thompson sum diverges)


## 6. From a mass to cells in a person

`q` is the selection constant carrying a generation probability to a post-selection repertoire
frequency. It defaults to `1`, i.e. **uncalibrated** — the returned `F` is then a raw model mass
whose ranking, not scale, is meaningful. ALICE's published TRB value is `9.41`; the matched
comparison against the model-free event ratio puts the same factor at a median 14.8×, so the two
are the same order and not the same number. Whatever you pass is echoed back, so a reported number
always carries its calibration.

`F(e)` answers "is there a precursor". A detectable response needs more than one, which is what
`p_ge_k` is for — and once $N_{\mathrm{eff}} F$ is of order one the two stop being a monotone
reparametrisation of each other.

In [8]:
# CD8 compartment of a 1e11-cell pool, and precursor probabilities at a realistic clonotype count.
out = P.precursor_frequency(model, junctions, r=1, q=P.ALICE_Q,
                            n_cells=1e11, compartment=0.3, n_eff=1e8)
print(f"F(e)              {out['F']:.3e}   (q = {out['q']}, alpha = {out['alpha']})")
print(f"expected cells    {out['cells']:,.0f}   of 1e11 x 0.3 CD8")
print(f"lambda            {out['lambda']:.3g}")
print(f"P(>= 1 precursor) {out['p_ge_1']:.4f}")
print(f"P(>= 10)          {out['p_ge_10']:.4f}")

F(e)              9.792e-03   (q = 9.41, alpha = 0.1)
expected cells    293,772,385   of 1e11 x 0.3 CD8
lambda            9.79e+05
P(>= 1 precursor) 1.0000
P(>= 10)          1.0000


## 7. The same thing from the command line

```console
$ vdjmatch precursor --vdjdb --mhc-class MHCI --min-junctions 10 --q 9.41 -o precursor.txt
```

writes one row per (epitope, chain) with every column above. `--group-by` and `--chain-col` do the
same for your own table.

## What this does not answer

`F(e)` is the probability a precursor **exists**, not that a response is **mounted**: that also
needs priming, help and an absence of tolerance, none of which is modelled here. And every set
total — the observed mass, the union, the event ratio alike — grows with how much of the true
cognate set the database holds, because that enters the numerator alone. A claim that ranks
epitopes by a set total needs that control stated alongside it.